# FracAtlas model comparison

This notebook trains three independent image classifiers on the same FracAtlas split. The candidate used by the FractureCare API is selected on the validation split by **fracture-focused macro recall** (the average recall of ONE_FRACTURE and MULTIPLE_FRACTURES), with all-class macro F1 as the tie-breaker. The selected model is then evaluated once on the untouched test split. This prioritises finding the two minority fracture categories in this imbalanced dataset; balanced accuracy, per-class metrics and the confusion matrix are also reported.

| Model | Why it is included |
| --- | --- |
| Custom CNN | A transparent, lightweight baseline that can be changed easily and establishes a reference point. |
| MobileNetV2 | An efficient transfer-learning model suited to limited data and practical CPU inference. |
| EfficientNetB0 | A stronger accuracy/efficiency candidate whose compound scaling often performs well on image classification. |

This comparison is an engineering experiment, not clinical validation.

In [ ]:
from pathlib import Path
import json
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "1")
import math
import sys

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
SERVICE_DIR = next(path for path in candidates if (path / 'app').is_dir() and (path / 'requirements.txt').exists())
PROJECT_DIR = SERVICE_DIR.parent
sys.path.insert(0, str(SERVICE_DIR))
from app.config import ARTIFACT_DIR, DATASET_CSV, IMAGE_DIR, IMAGE_SIZE, SEED
from app.data import load_manifest
from app.labels import CLASS_NAMES
from app.metrics import SELECTION_METRIC, calculate_classification_metrics, select_best_result
from app.model import MODEL_NAMES, build_model, build_transfer_model

MODEL_EPOCHS = 15
BATCH_SIZE = 32
USE_IMAGENET_WEIGHTS = True  # Set to False for an offline comparison.
MODEL_DIR = ARTIFACT_DIR / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
tf.keras.utils.set_random_seed(SEED)
print('TensorFlow:', tf.__version__)
print('Dataset:', DATASET_CSV)

In [ ]:
frame = load_manifest()
train, holdout = train_test_split(frame, test_size=0.2, random_state=SEED, stratify=frame['label_index'])
validation, test = train_test_split(holdout, test_size=0.5, random_state=SEED, stratify=holdout['label_index'])
train, validation, test = [part.reset_index(drop=True) for part in (train, validation, test)]
print(f'Images: {len(frame):,} | train: {len(train):,} | validation: {len(validation):,} | test: {len(test):,}')
display(frame['label'].value_counts().reindex(CLASS_NAMES).rename('count').to_frame())

In [ ]:
def make_dataset(dataframe, shuffle=False):
    paths = dataframe['path'].to_numpy()
    labels = dataframe['label_index'].to_numpy(dtype=np.int32)
    def load(path, label):
        image = tf.io.read_file(path)
        image = tf.io.decode_jpeg(image, channels=3)
        image = tf.image.resize(image, IMAGE_SIZE)
        return image, label
    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        dataset = dataset.shuffle(len(dataframe), seed=SEED, reshuffle_each_iteration=True)
    dataset = dataset.map(load, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.apply(tf.data.experimental.ignore_errors())
    return dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_dataset = make_dataset(train, shuffle=True).repeat()
validation_dataset = make_dataset(validation).repeat()
validation_eval_dataset = make_dataset(validation)
test_dataset = make_dataset(test)
TRAIN_STEPS = math.ceil(len(train) / BATCH_SIZE)
VALIDATION_STEPS = math.ceil(len(validation) / BATCH_SIZE)
weights = compute_class_weight('balanced', classes=np.arange(len(CLASS_NAMES)), y=train['label_index'].to_numpy())
class_weights = {index: float(weight) for index, weight in enumerate(weights)}
print('Class weights:', class_weights)

## Train each model separately

Each loop iteration creates a fresh model, trains it from the same training split, selects its best validation checkpoint, and evaluates it only once on the held-out test split. ImageNet weights are used only by the two transfer-learning models.

In [ ]:
def create_model(model_name):
    if model_name == 'custom_cnn':
        return build_model()
    weights = 'imagenet' if USE_IMAGENET_WEIGHTS else None
    return build_transfer_model(model_name, weights=weights)

model_paths = {}
histories = {}
results = []

for model_name in MODEL_NAMES:
    print(f'\n===== Training {model_name} =====')
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(SEED)
    model = create_model(model_name)
    path = MODEL_DIR / f'{model_name}.keras'
    callbacks = [
        tf.keras.callbacks.ModelCheckpoint(path, monitor='val_accuracy', save_best_only=True),
        tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=2, min_lr=1e-6),
    ]
    history = model.fit(train_dataset, validation_data=validation_dataset, epochs=MODEL_EPOCHS, steps_per_epoch=TRAIN_STEPS, validation_steps=VALIDATION_STEPS, class_weight=class_weights, callbacks=callbacks, shuffle=False)
    histories[model_name] = history.history
    model_paths[model_name] = path
    del model

In [ ]:
for model_name, path in model_paths.items():
    model = tf.keras.models.load_model(path)
    probabilities = model.predict(validation_eval_dataset, verbose=0)
    predicted = probabilities.argmax(axis=1)
    actual = np.concatenate([labels.numpy() for _, labels in validation_eval_dataset], axis=0)
    metrics = calculate_classification_metrics(actual, predicted)
    results.append({'model': model_name, **metrics})
    print(f'\n{model_name}')
    print(classification_report(actual, predicted, target_names=CLASS_NAMES, zero_division=0))

results_df = pd.DataFrame(results).sort_values([SELECTION_METRIC, 'macro_f1', 'balanced_accuracy'], ascending=False).reset_index(drop=True)
display(results_df.style.format({column: '{:.4f}' for column in results_df.columns if column != 'model'}))

In [ ]:
best_result = select_best_result(results)
best_name = best_result['model']
best_path = model_paths[best_name]
best_model = tf.keras.models.load_model(best_path)
final_probabilities = best_model.predict(test_dataset, verbose=0)
final_predicted = final_probabilities.argmax(axis=1)
final_actual = np.concatenate([labels.numpy() for _, labels in test_dataset], axis=0)
final_test_metrics = calculate_classification_metrics(final_actual, final_predicted)
print('Final untouched test-set metrics:')
print(classification_report(final_actual, final_predicted, target_names=CLASS_NAMES, zero_division=0))
best_model.save(ARTIFACT_DIR / 'fracture_classifier.keras')
results_df.to_csv(ARTIFACT_DIR / 'model_comparison.csv', index=False)
test.to_csv(ARTIFACT_DIR / 'test_manifest.csv', index=False)
metadata = {
    'modelVersion': f'fracatlas-{best_name}-1.0.0',
    'selectedModel': best_name,
    'classes': list(CLASS_NAMES),
    'imageSize': list(IMAGE_SIZE),
    'datasetCsv': str(DATASET_CSV),
    'trainCount': int(len(train)),
    'validationCount': int(len(validation)),
    'testCount': int(len(test)),
    'selectionMetric': SELECTION_METRIC,
    'comparison': results_df.to_dict(orient='records'),
    'finalTestMetrics': final_test_metrics,
}
(ARTIFACT_DIR / 'model_metadata.json').write_text(json.dumps(metadata, indent=2, default=float), encoding='utf-8')
print(f'Selected {best_name} using {SELECTION_METRIC} (macro F1 tie-breaker) and saved it to {ARTIFACT_DIR / "fracture_classifier.keras"}')
display(results_df)